Scenario: AI-Powered HR Onboarding System (Corporate HR Team)
🎯 Problem Statement
In companies:
- HR teams manually prepare onboarding documents and checklists.
- Managers validate compliance with policies.
- Employees often face delays in receiving resources and training schedules.
👉 Solution
A multi-agent system where:
- 📝 HR Agent prepares onboarding documents, training schedules, and resource lists.
- ✅ Compliance Agent reviews documents for policy adherence and legal requirements.
- 📂 Resource Agent allocates accounts, devices, and access permissions.
- 🔁 The loop continues until all onboarding steps are validated and complete.
🔄 Workflow
Onboarding Request → HR Agent drafts documents → Compliance Agent validates → Resource Agent provisions accounts/devices → Iteration loop → Final approved onboarding package
🧹 Conflict Resolution
- Resolve conflicting policy requirements (e.g., different regional compliance rules).
- Standardize onboarding templates across departments.
- Ensure system avoids duplicate resource allocation (e.g., multiple accounts for same employee).

In [1]:
# ============================================================
# Scenario: AI-Powered HR Onboarding System (Corporate HR Team)
# ============================================================

import os
import json
import re
from google.colab import userdata
from openai import OpenAI

# ------------------------------------------------------------
# 1. API Setup (Groq via OpenAI-compatible client)
# ------------------------------------------------------------
groq_api_key = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = groq_api_key

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY not found in Colab secrets. "
        "Add it in the Secrets panel with key name: GROQ_API_KEY"
    )

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = "llama-3.3-70b-versatile"

# ------------------------------------------------------------
# 2. Onboarding Request
# ------------------------------------------------------------
onboarding_request = """
Prepare a complete onboarding package for a new employee.

Employee details:
- Name: Aarav Sharma
- Employee ID: EMP1045
- Role: Data Analyst
- Department: Business Intelligence
- Region: India
- Work Mode: Hybrid
- Manager: Priya Mehta
- Joining Date: 2026-04-01

Requirements:
1. Prepare onboarding documents
2. Create training schedule
3. Prepare resource checklist
4. Validate compliance with company policy and regional requirements
5. Provision accounts, devices, and permissions
6. Avoid duplicate resource allocation
7. Standardize onboarding format across departments
"""

# ------------------------------------------------------------
# 3. Agent Prompts
# ------------------------------------------------------------
HR_AGENT_SYSTEM = """
You are the HR Agent.

Your responsibilities:
- Prepare a complete onboarding package
- Draft onboarding documents
- Create a training schedule
- Create a resource checklist
- Keep the format professional and standardized
- Resolve department formatting differences by using one common structure
- Do not return JSON
- Do not use markdown fences

Return exactly in this tag format:

<<ONBOARDING_DOCUMENT>>
full onboarding document here
<</ONBOARDING_DOCUMENT>>

<<TRAINING_SCHEDULE>>
full training schedule here
<</TRAINING_SCHEDULE>>

<<RESOURCE_CHECKLIST>>
full resource checklist here
<</RESOURCE_CHECKLIST>>
"""

COMPLIANCE_AGENT_SYSTEM = """
You are the Compliance Agent.

Your responsibilities:
- Review onboarding materials for policy adherence
- Check legal and regional compliance
- Resolve regional policy conflicts with the safest compliant option
- Ensure standardized onboarding templates
- Do not return JSON
- Do not use markdown fences

Return exactly in this tag format:

<<APPROVED>>
YES or NO
<</APPROVED>>

<<COMPLIANCE_NOTES>>
note 1
note 2
<</COMPLIANCE_NOTES>>

<<REQUIRED_FIXES>>
fix 1
fix 2
<</REQUIRED_FIXES>>
"""

RESOURCE_AGENT_SYSTEM = """
You are the Resource Agent.

Your responsibilities:
- Provision accounts, devices, and permissions
- Avoid duplicate resource allocation
- Ensure role-based access is practical and minimal
- Prepare final allocation summary
- Do not return JSON
- Do not use markdown fences

Return exactly in this tag format:

<<RESOURCE_ALLOCATION>>
full resource allocation details here
<</RESOURCE_ALLOCATION>>

<<DUPLICATE_CHECK>>
PASS or FAIL
<</DUPLICATE_CHECK>>

<<RESOURCE_ISSUES>>
issue 1
issue 2
<</RESOURCE_ISSUES>>
"""

# ------------------------------------------------------------
# 4. Helper Functions
# ------------------------------------------------------------
def call_llm(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content.strip()


def extract_tag(text, tag_name, required=True):
    pattern = rf"<<{tag_name}>>\s*(.*?)\s*<</{tag_name}>>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    if required:
        raise ValueError(f"Missing tag: {tag_name}\n\nRaw response:\n{text}")
    return ""


def parse_lines_block(block_text):
    lines = []
    for line in block_text.splitlines():
        cleaned = line.strip()
        if cleaned:
            lines.append(cleaned)
    return lines


def parse_hr_response(text):
    return {
        "onboarding_document": extract_tag(text, "ONBOARDING_DOCUMENT"),
        "training_schedule": extract_tag(text, "TRAINING_SCHEDULE"),
        "resource_checklist": extract_tag(text, "RESOURCE_CHECKLIST")
    }


def parse_compliance_response(text):
    approved_text = extract_tag(text, "APPROVED").strip().upper()
    approved = approved_text in {"YES", "TRUE", "APPROVED"}

    notes_block = extract_tag(text, "COMPLIANCE_NOTES", required=False)
    fixes_block = extract_tag(text, "REQUIRED_FIXES", required=False)

    return {
        "approved": approved,
        "compliance_notes": parse_lines_block(notes_block) if notes_block else [],
        "required_fixes": parse_lines_block(fixes_block) if fixes_block else []
    }


def parse_resource_response(text):
    issues_block = extract_tag(text, "RESOURCE_ISSUES", required=False)

    return {
        "resource_allocation": extract_tag(text, "RESOURCE_ALLOCATION"),
        "duplicate_check": extract_tag(text, "DUPLICATE_CHECK").strip().upper(),
        "resource_issues": parse_lines_block(issues_block) if issues_block else []
    }


# ------------------------------------------------------------
# 5. Iterative Multi-Agent Workflow
# ------------------------------------------------------------
max_iterations = 5
hr_output = None
compliance_output = None
resource_output = None
feedback = "No previous feedback."

for iteration in range(1, max_iterations + 1):
    print("\n" + "=" * 70)
    print(f"ITERATION {iteration}")
    print("=" * 70)

    # ---------------- HR Agent ----------------
    hr_prompt = f"""
Onboarding Request:
{onboarding_request}

Previous feedback:
{feedback}

Prepare the onboarding package.
Use the required tag format exactly.
"""
    hr_raw = call_llm(HR_AGENT_SYSTEM, hr_prompt)
    hr_output = parse_hr_response(hr_raw)

    print("\nHR Agent completed onboarding package draft.")

    # ---------------- Compliance Agent ----------------
    compliance_prompt = f"""
Onboarding Request:
{onboarding_request}

Onboarding Document:
{hr_output['onboarding_document']}

Training Schedule:
{hr_output['training_schedule']}

Resource Checklist:
{hr_output['resource_checklist']}

Review for policy adherence, legal alignment, regional conflicts, and standardization.
Use the required tag format exactly.
"""
    compliance_raw = call_llm(COMPLIANCE_AGENT_SYSTEM, compliance_prompt)
    compliance_output = parse_compliance_response(compliance_raw)

    print("Compliance Agent completed review.")
    print(json.dumps(compliance_output, indent=2, ensure_ascii=False))

    # ---------------- Resource Agent ----------------
    resource_prompt = f"""
Onboarding Request:
{onboarding_request}

Role:
Data Analyst

Department:
Business Intelligence

Onboarding Document:
{hr_output['onboarding_document']}

Resource Checklist:
{hr_output['resource_checklist']}

Provision resources, avoid duplicates, and generate final resource allocation details.
Use the required tag format exactly.
"""
    resource_raw = call_llm(RESOURCE_AGENT_SYSTEM, resource_prompt)
    resource_output = parse_resource_response(resource_raw)

    print("Resource Agent completed provisioning simulation.")
    print(json.dumps(resource_output, indent=2, ensure_ascii=False))

    compliance_ok = compliance_output.get("approved", False)
    duplicate_ok = resource_output.get("duplicate_check", "") == "PASS"

    if compliance_ok and duplicate_ok:
        print("\nFinal status: APPROVED")
        break

    combined_feedback = (
        compliance_output.get("required_fixes", []) +
        resource_output.get("resource_issues", [])
    )

    if combined_feedback:
        feedback = "Please revise using this feedback:\n" + "\n".join(
            [f"- {item}" for item in combined_feedback]
        )
    else:
        feedback = "Please improve compliance coverage, standardization, and resource allocation accuracy."

# ------------------------------------------------------------
# 6. Save Final Onboarding Package
# ------------------------------------------------------------
final_package = {
    "onboarding_request": onboarding_request.strip(),
    "hr_output": hr_output,
    "compliance_output": compliance_output,
    "resource_output": resource_output
}

with open("onboarding_package.json", "w", encoding="utf-8") as f:
    json.dump(final_package, f, indent=2, ensure_ascii=False)

with open("onboarding_package.txt", "w", encoding="utf-8") as f:
    f.write("AI-Powered HR Onboarding System\n")
    f.write("=" * 60 + "\n\n")
    f.write("ONBOARDING REQUEST\n")
    f.write("-" * 60 + "\n")
    f.write(onboarding_request.strip() + "\n\n")

    f.write("ONBOARDING DOCUMENT\n")
    f.write("-" * 60 + "\n")
    f.write(hr_output["onboarding_document"] + "\n\n")

    f.write("TRAINING SCHEDULE\n")
    f.write("-" * 60 + "\n")
    f.write(hr_output["training_schedule"] + "\n\n")

    f.write("RESOURCE CHECKLIST\n")
    f.write("-" * 60 + "\n")
    f.write(hr_output["resource_checklist"] + "\n\n")

    f.write("COMPLIANCE REVIEW\n")
    f.write("-" * 60 + "\n")
    f.write(json.dumps(compliance_output, indent=2, ensure_ascii=False) + "\n\n")

    f.write("RESOURCE ALLOCATION\n")
    f.write("-" * 60 + "\n")
    f.write(resource_output["resource_allocation"] + "\n\n")

    f.write("RESOURCE VALIDATION\n")
    f.write("-" * 60 + "\n")
    f.write(json.dumps(resource_output, indent=2, ensure_ascii=False) + "\n")

print("\nSaved files:")
print("- onboarding_package.json")
print("- onboarding_package.txt")

print("\n" + "=" * 70)
print("FINAL HR OUTPUT")
print("=" * 70)
print(json.dumps(hr_output, indent=2, ensure_ascii=False))

print("\n" + "=" * 70)
print("FINAL COMPLIANCE OUTPUT")
print("=" * 70)
print(json.dumps(compliance_output, indent=2, ensure_ascii=False))

print("\n" + "=" * 70)
print("FINAL RESOURCE OUTPUT")
print("=" * 70)
print(json.dumps(resource_output, indent=2, ensure_ascii=False))

print("\nDONE")


ITERATION 1

HR Agent completed onboarding package draft.
Compliance Agent completed review.
{
  "approved": true,
  "compliance_notes": [
    "The onboarding package is comprehensive and adheres to company policies and procedures.",
    "The training schedule is well-structured and covers all necessary aspects of the role.",
    "The resource checklist is thorough and ensures that the new employee has all the necessary tools and support.",
    "The package is standardized across departments, ensuring consistency and compliance."
  ],
  "required_fixes": [
    "None"
  ]
}
Resource Agent completed provisioning simulation.
{
  "resource_allocation": "Employee Name: Aarav Sharma\nEmployee ID: EMP1045\nRole: Data Analyst\nDepartment: Business Intelligence\nRegion: India\nWork Mode: Hybrid\nManager: Priya Mehta\nJoining Date: 2026-04-01\nHardware: Laptop, Mobile Phone, Headset\nSoftware: Microsoft Office, Data Analysis Tools, Data Visualization Software\nAccounts and Permissions: Email Ac